# 第 8 章 · 中间件、发现与网络

本章回答：同一个 Wi-Fi 下两组人都在跑 ROS 2，怎么不互相串台？机器人和笔记本为什么有时能互相看见、有时不能？DDS 和 Zenoh 有什么不同，怎么换？`ros2 node list` 为什么有时会显示已经关掉的节点？

配套讲义：仓库里 `ros2/notes/lessons/08-middleware-discovery.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 DDS。端口公式只是加减乘，官方文档 About-Domain-ID.rst L88–L110 给了一串例子，这里逐个算出来核对。公式里的 7400、250、10、2 是 DDS 规范（RTPS）的默认常数，文档本身只链接了外部说明。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 组播端口 | 7400 + 250 × 域号 | `multicast_ports(1)` | (7650, 7651)，文档 L92 |
| 第 1 步 单播端口 | 每多一个进程往后挪 2 个 | `unicast_ports(1, 120)` | (7900, 7901)，撞上域 2，文档 L96 |
| 第 1 步 域号上限 | 端口最大 65535 | `max_domain()` | 232，文档 L50 |
| 第 1 步 Linux 0–101 | 避开临时端口段 32768–60999 | `unicast_ports(101, 54)` | (32768, 32769)，文档 L107 |
| 同上 | 域 101 / 232 最多几个进程 | `max_participants(...)` | 54 / 63，文档 L109–L110 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `LINUX_EPHEMERAL`**

In [ ]:
LINUX_EPHEMERAL = (32768, 60999)  # About-Domain-ID.rst L65：Linux 默认临时端口段

**常量 `MAX_PORT`**

In [ ]:
MAX_PORT = 65535

**常量 `PB`, `DG`, `PG`, `D_UNICAST`**

In [ ]:
PB, DG, PG, D_UNICAST = 7400, 250, 2, 10  # 端口基数、每个域占的端口段长、每个参与者占几个端口、单播偏移

**函数 `max_domain`**：组播端口不能超过 65535：7400 + 250 × d ≤ 65535。

In [ ]:
def max_domain() -> int:
    """组播端口不能超过 65535：7400 + 250 × d ≤ 65535。"""
    return (MAX_PORT - PB) // DG

**函数 `unicast_ports`**：同一台电脑上第 participant 个 ROS 2 进程（从 0 数）用的两个单播端口。

In [ ]:
def unicast_ports(domain: int, participant: int) -> tuple[int, int]:
    """同一台电脑上第 participant 个 ROS 2 进程（从 0 数）用的两个单播端口。"""
    base = PB + DG * domain + D_UNICAST + PG * participant
    return base, base + 1

**函数 `max_participants`**：单播端口不越过 limit（临时端口段起点或 65535）时，最多能开几个进程。

In [ ]:
def max_participants(domain: int, limit: int) -> int:
    """单播端口不越过 limit（临时端口段起点或 65535）时，最多能开几个进程。"""
    p = 0
    while unicast_ports(domain, p)[1] < limit:
        p += 1
    return p

**函数 `multicast_ports`**：一个域的发现组播端口和用户数据组播端口。

In [ ]:
def multicast_ports(domain: int) -> tuple[int, int]:
    """一个域的发现组播端口和用户数据组播端口。"""
    base = PB + DG * domain
    return base, base + 1

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 组播端口：7400 + 250 × 域号（文档 L92–L93）")
for d in [0, 1, 2, 7]:
    print(f"  域 {d}: 组播端口 {multicast_ports(d)}")

In [ ]:
print("\n第一步（续）· 单播端口：每多一个进程往后挪 2 个（文档 L94–L96）")
for p in [0, 119, 120]:
    print(f"  域 1 第 {p} 号参与者: {unicast_ports(1, p)}")
print("  域 1 的第 120 号已经撞上域 2 的组播端口", multicast_ports(2))

In [ ]:
print("\n第一步（再续）· 域号上限：", max_domain(), "（文档 L50）")

In [ ]:
print("\n第一步（又续）· Linux 上域 101：第 53 号和第 54 号进程（文档 L102–L107）")
print("  第 53 号:", multicast_ports(101), unicast_ports(101, 53))
print("  第 54 号:", multicast_ports(101), unicast_ports(101, 54), "← 进了临时端口段", LINUX_EPHEMERAL)
print("  域 101 最多开", max_participants(101, LINUX_EPHEMERAL[0]), "个进程；域 232 最多开",
      max_participants(232, MAX_PORT + 1), "个（文档 L109–L110）")

## 3. 代码里才看得到的

1. **讲义只说了"域号算出端口，上限 232"**，公式在官方文档链接的外部文章里。这里的四个常数取自 DDS 线上协议 RTPS 的默认值，测试拿 `About-Domain-ID.rst` L92–L110 给出的每一个例子逐个核对，全部吻合。
2. **"安全"的意思是组播端口不进临时端口段**：域 101 的组播端口 32650 在 32768 以下，但开到第 54 号进程时单播端口就进去了，所以文档说域 101 最多 54 个进程。
3. **Zenoh 不用这套端口**：它的路由器监听 TCP 7447（第 2 步），和域号算端口无关。

## 4. 自己改一改

- 算一算你常用的域号（比如 7、42）的组播端口；
- 把 `LINUX_EPHEMERAL` 换成 macOS 的 (49152, 65535)，用 `multicast_ports` 验证文档 L73 说的"macOS 上 0–166 安全"。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **分组**：同号同频道，默认是 0；安全取 0 到 101。
- **范围**：LOCALHOST 只看本机，STATIC_PEERS 点名找人。
- **中间件**：一个环境变量换插头，全系统用同一个；Zenoh 先起路由器。
- **排错**：域号、中间件、范围、ping、组播、QoS；缓存不对 daemon stop。